# Path B — GRPO v2 twins: real reward vs random reward (seed 3407)

**Runtime: A100.** ~2.5 h per run, TWO runs (~60 units total). The RUNS list is
checkpointed — run 'real' first, stop, run 'random' in a later session if you
prefer. Both runs are byte-identical except the reward.

**The scientific question** (from the v1 control, S2.19): at v0 data difficulty,
GRPO's gain was fully reproduced by a coin-flip reward — process, not signal.
Now everything the signal lacked is fixed: **harder data** (v1 pile, the v2
model still fails ~26% of first tries), **2× steps (500)**, **2× lr (1e-5)**,
and an **edit-aware penalty** (QiMeng EA-GRPO: among CORRECT samples in mostly-
solving groups, over-editing is penalized — capped at 0.10 so the CoRPO
invariant 1.15 > 0.30 still holds).

**Pre-registered predictions:**
- real ≫ random → the execution signal becomes causal when it has leverage
- real ≈ random > SFT v2 → the process effect persists, signal still inert
- both ≈ SFT v2 → RL saturated at this scale regardless of reward

Dev evals here are PRELIMINARY (60 new-dev bugs, noisy ±3-4); the exam
(notebook 18) is the referee. Init: merged SFT v2 s3407 ep2 + fresh LoRA.

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib, gc, uuid
PHASE8 = '/content/drive/MyDrive/rl-post-training/phase8'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts', 'reward', 'variance_gate'):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
import reward as rw
import variance_gate as vg
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
v1_bugs = json.load(open(f'{PHASE8}/data_v1_bugs.json'))
v0_restraint = json.load(open('/content/ptl/data/data_v0_restraint.json'))
train_v1 = [b for b in v1_bugs if b['split'] == 'train']
dev_new = [b for b in v1_bugs if b['split'] == 'dev' and b.get('gen') == 'deepseek_v1']
dev_new_sample = random.Random(3407).sample(dev_new, 60)   # the standard 60
dev_clean = [r for r in v0_restraint if r['split'] == 'dev']
SEED = 3407
RUNS = ['real', 'random']   # checkpointed; trim to ['real'] to split sessions
print(len(train_v1), 'train bugs |', len(dev_new_sample), 'dev |', len(dev_clean), 'probe')

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# Hardened runner + BOTH rewards (real w/ edit-aware penalty; random twin)
import subprocess, tempfile, difflib
from concurrent.futures import ThreadPoolExecutor
SENTINEL = uuid.uuid4().hex
G, ALPHA, CAP = 8, 0.8, 0.10   # group size, EA accuracy threshold, EA penalty cap
WATCH = ['']
_rng = random.Random(SEED)
_call_n = [0]

def run_tests(code, rec, timeout=10):
    tests = list(rec['test_list'])
    try:
        compile(code, '<cand>', 'exec'); compiles = True
    except Exception:
        return (False, 0, len(tests))
    harness = (
        '\n'.join(list(rec['test_imports'])) + '\n' + code + '\n'
        + '__zz_pass = 0\n'
        + f'for __zz_t in {tests!r}:\n'
        + '    try:\n        exec(__zz_t)\n        __zz_pass += 1\n'
        + '    except BaseException:\n        pass\n'
        + f'print("{SENTINEL}", __zz_pass)\n'
    )
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(harness); path = f.name
    try:
        r = subprocess.run([sys.executable, path], capture_output=True, timeout=timeout)
        n = 0
        for line in r.stdout.decode(errors='ignore').splitlines():
            parts = line.split()
            if len(parts) == 2 and parts[0] == SENTINEL and parts[1].isdigit():
                n = int(parts[1])
    except subprocess.TimeoutExpired:
        n = 0
    finally:
        os.unlink(path)
    return (compiles, n, len(tests))

def edit_cost(buggy, fix):
    return 1.0 - difflib.SequenceMatcher(None, buggy, fix).ratio()

def grade_batch(completions, test_imports, test_list):
    recs = [{'test_imports': ti, 'test_list': tl} for ti, tl in zip(test_imports, test_list)]
    codes = [extract_code(c) for c in completions]
    with ThreadPoolExecutor(max_workers=8) as pool:
        outs = list(pool.map(lambda a: run_tests(*a), zip(codes, recs)))
    return codes, outs

def reward_real(prompts, completions, test_imports, test_list, buggy, **kw):
    codes, outs = grade_batch(completions, test_imports, test_list)
    rewards = [rw.reward(rw.ExecResult(compiles=c, num_passed=p, num_tests=t,
                                       output_tokens=len(comp) // 4))
               for (c, p, t), comp in zip(outs, completions)]
    pen_applied = 0.0
    for s in range(0, len(rewards), G):   # edit-aware penalty per group
        sl = slice(s, s + G)
        grp = list(range(s, min(s + G, len(rewards))))
        passing = [i for i in grp if outs[i][1] == outs[i][2]]
        if len(passing) / max(len(grp), 1) < ALPHA or len(passing) < 2:
            continue
        costs = {i: edit_cost(buggy[i], codes[i]) for i in passing}
        lo, hi = min(costs.values()), max(costs.values())
        if hi - lo < 1e-9:
            continue
        for i in passing:
            pen = CAP * (costs[i] - lo) / (hi - lo)
            rewards[i] -= pen
            pen_applied += pen
    _call_n[0] += 1
    with open(WATCH[0], 'a') as f:
        f.write(json.dumps({'call': _call_n[0], 'mean_r': sum(rewards)/len(rewards),
                            'n_pass': sum(1 for c, p, t in outs if p == t),
                            'pen_applied': round(pen_applied, 4)}) + '\n')
    return rewards

def reward_random(prompts, completions, test_imports, test_list, buggy, **kw):
    rewards = [_rng.uniform(0.0, 1.3) for _ in completions]
    codes, outs = grade_batch(completions, test_imports, test_list)   # silent truth
    _call_n[0] += 1
    with open(WATCH[0], 'a') as f:
        f.write(json.dumps({'call': _call_n[0], 'true_pass_frac':
                            sum(1 for c, p, t in outs if p == t)/len(outs)}) + '\n')
    return rewards

# self-test: hack must score 0 passed; minimal fix must out-reward over-edit
_demo = {'test_imports': [], 'test_list': ['assert add2(2) == 4']}
_c, _p, _t = run_tests('import sys\nsys.exit(0)\ndef add2(x):\n    return x', _demo)
print('HACK sys.exit(0): passed =', _p, '(must be 0)')
print('edit_cost minimal vs rewrite:', round(edit_cost('def f(x):\n    return x+1', 'def f(x):\n    return x+2'), 3),
      'vs', round(edit_cost('def f(x):\n    return x+1', 'def g(y):\n    z = y\n    return z + 2'), 3))

In [ ]:
# Eval helpers + init merge (unsloth ok here — this env trained nb 15 fine)
import torch
from unsloth import FastLanguageModel
try:
    from unsloth import PatchFastRL
    PatchFastRL('GRPO', FastLanguageModel)
except ImportError:
    pass
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

def chat(tok, p):
    return tok.apply_chat_template([{'role': 'user', 'content': p}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def sample_k(model, tok, source_code, test_list, k):
    enc = tok([chat(tok, build_repair_prompt(source_code, test_list))],
              return_tensors='pt', padding=True, padding_side='left').to('cuda')
    out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                         num_return_sequences=k, max_new_tokens=512,
                         pad_token_id=tok.eos_token_id)
    return [extract_code(t) for t in tok.batch_decode(
        out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]

def passes_all(code, rec):
    c, p, t = run_tests(code, rec)
    return p == t

def dev_eval(model, tok, tag, k=8):
    FastLanguageModel.for_inference(model)
    per_bug = []
    for b in dev_new_sample:
        codes = sample_k(model, tok, b['buggy'], b['test_list'], k)
        with ThreadPoolExecutor(max_workers=8) as pool:
            flags = list(pool.map(lambda c: passes_all(c, b), codes))
        per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    pk = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    json.dump({'tag': tag, 'pass@1': p1, 'pass@8': pk, 'per_bug': per_bug},
              open(f'{PHASE8}/dev_eval_{tag}.json', 'w'))
    print(f"[{tag}]  pass@1={p1*100:.1f}%  pass@8={pk*100:.1f}%")

def restraint_probe(model, tok, tag, k=4):
    FastLanguageModel.for_inference(model)
    norm = lambda s: ' '.join(s.split())
    sp = un = tot = 0
    for r in dev_clean:
        codes = sample_k(model, tok, r['code'], r['test_list'], k)
        with ThreadPoolExecutor(max_workers=8) as pool:
            flags = list(pool.map(lambda c: passes_all(c, r), codes))
        sp += sum(flags); un += sum(norm(c) == norm(r['code']) for c in codes); tot += k
    print(f"[probe {tag}]  still passes: {sp/tot*100:.1f}%  unchanged: {un/tot*100:.1f}%")

MERGED = '/content/merged_sftv2_s3407'
if not os.path.exists(MERGED):
    m, t = FastLanguageModel.from_pretrained(
        f'{PHASE8}/sft_v2_s3407_ep2', max_seq_length=1536,
        load_in_4bit=False, dtype=torch.bfloat16)
    m.save_pretrained_merged(MERGED, t, save_method='merged_16bit')
    del m, t; gc.collect(); torch.cuda.empty_cache()
print('init ready:', MERGED)

In [ ]:
# Advisory gate on the v2 init (40 random v1 train bugs x 8)
model, tok = FastLanguageModel.from_pretrained(
    MERGED, max_seq_length=1536, load_in_4bit=True, dtype=None)
FastLanguageModel.for_inference(model)
pc = []
for i, b in enumerate(random.Random(SEED).sample(train_v1, 40)):
    codes = sample_k(model, tok, b['buggy'], b['test_list'], 8)
    with ThreadPoolExecutor(max_workers=8) as pool:
        outs = list(pool.map(lambda c: passes_all(c, b), codes))
    pc.append((b['id'], sum(outs)))
    if (i + 1) % 10 == 0:
        print(f'{i+1}/40')
g = vg.gate(pc, group_size=8)
print(f"\nGATE (v2 init on v1 pile): learnable {g['learnable_fraction']*100:.0f}% (needs >=30%)")
del model, tok; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# THE TWIN RUNS — identical except the reward function
def run_twin(kind):
    print(f'===== GRPO v2 [{kind}] seed {SEED} — 500 steps, lr 1e-5 =====')
    WATCH[0] = f'{PHASE8}/watch_grpo_v2_{kind}_s{SEED}.jsonl'
    _call_n[0] = 0
    model, tok = FastLanguageModel.from_pretrained(
        MERGED, max_seq_length=1536, load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=32, lora_alpha=64, lora_dropout=0,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        use_gradient_checkpointing='unsloth', random_state=SEED)
    rows = [{'prompt': chat(tok, build_repair_prompt(b['buggy'], b['test_list'])),
             'test_imports': list(b['test_imports']), 'test_list': list(b['test_list']),
             'buggy': b['buggy']} for b in train_v1]
    ds = Dataset.from_list(rows).shuffle(seed=SEED)
    FastLanguageModel.for_training(model)
    cfg = GRPOConfig(
        per_device_train_batch_size=8, gradient_accumulation_steps=4,
        num_generations=8, max_prompt_length=896, max_completion_length=512,
        temperature=1.0, top_p=0.95, beta=0.04,
        learning_rate=1e-5, lr_scheduler_type='cosine', warmup_ratio=0.1,
        max_steps=500, logging_steps=20, seed=SEED, output_dir='/content/out',
        report_to='none', save_strategy='no')
    fn = reward_real if kind == 'real' else reward_random
    trainer = GRPOTrainer(model=model, processing_class=tok, reward_funcs=fn,
                          args=cfg, train_dataset=ds)
    trainer.train()
    model.save_pretrained(f'{PHASE8}/grpo_v2_{kind}_s{SEED}')
    dev_eval(model, tok, f'grpo_v2_{kind}_newdev_seed{SEED}')
    restraint_probe(model, tok, f'grpo_v2_{kind}')
    del trainer, model, tok
    gc.collect(); torch.cuda.empty_cache()

for kind in RUNS:
    if os.path.exists(f'{PHASE8}/dev_eval_grpo_v2_{kind}_newdev_seed{SEED}.json'):
        print(kind, 'already done, skipping'); continue
    run_twin(kind)

In [ ]:
# PRELIMINARY VERDICT (dev; the exam in notebook 18 is the referee)
def load(tag):
    p = f'{PHASE8}/dev_eval_{tag}.json'
    return json.load(open(p)) if os.path.exists(p) else None
sft = load('sftv2_ep2_newdev_seed3407')
real = load(f'grpo_v2_real_newdev_seed{SEED}')
rand = load(f'grpo_v2_random_newdev_seed{SEED}')
print(f"{'model (s3407, new-dev 60)':<28} pass@1   pass@8")
for name, r in (('SFT v2 (init)', sft), ('GRPO v2 REAL reward', real),
                ('GRPO v2 RANDOM reward', rand)):
    if r:
        print(f"{name:<28} {r['pass@1']*100:6.1f}%  {r['pass@8']*100:6.1f}%")
    else:
        print(f'{name:<28} (not run)')
print('\nPre-registered readings: real >> random = signal causal on harder data;')
print('real ~ random > SFT = process effect persists; both ~ SFT = RL saturated.')
print('Dev noise here is +/-3-4 pts — the exam (notebook 18) decides. Also check')
print('the real-run watch file: pen_applied trend = is the EA penalty firing?')

## Bring back to the session
1. The **gate line** (learnable % of the v1 pile for the v2 init)
2. A few **reward-table rows** from each run (start/middle/end)
3. The **preliminary verdict table** + both **probe lines**
4. First/last lines of both **watch files**

Then notebook 18: one exam run each for the real and random models — the
attribution referee, and the shot at GPT-4's 47.